# VOC Intelligence — GraphRAG 튜토리얼 (Phase 3)

이 노트북은 `src/graph/*`의 **실제 모듈을 셀에서 import·실행**하며 VOC 코파일럿의
핵심 5개 개념을 설명합니다 (PHASE3_PLAN §3.3).

1. **온톨로지** — 지식그래프 스키마 (Customer→Conversation→Symptom→Component→RootCause→Action)
2. **3중 하이브리드 검색** — dense(bge-small) + sparse(full-text) + graph(structural)
3. **RRF 융합** — Reciprocal Rank Fusion
4. **retrieval-gating** — 3분기 정직 게이트 (refuse / low_confidence / answer)
5. **LangGraph 에이전트 + HITL interrupt** — 사람 승인 후 이슈 디스패치

> **실행 안내.** 아래 `LIVE = False`가 기본값입니다. 이 상태에서는 Neo4j·LLM을 건드리는
> 셀이 스킵되어 **전체 노트북이 인프라 없이 에러 없이 실행**됩니다. 실제 그래프/LLM 왕복을
> 보려면 `.env`(NEO4J_*, OPENROUTER_API_KEY)를 채우고 `LIVE = True`로 바꾸세요.

In [1]:
# ── 셋업: repo 루트를 sys.path에 올려 `src.graph.*` import를 가능하게 한다 ──
import sys, pathlib

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# 라이브(Neo4j Aura + OpenRouter) 셀 실행 여부. 기본 False → 인프라 없이 통과.
LIVE = False

from src.graph import config
print("repo root :", ROOT)
print("LIVE      :", LIVE)
print("임계값     :", config.ROOTCAUSE_MIN_CONVERSATIONS, "(ROOTCAUSE_MIN_CONVERSATIONS)")

repo root : /Users/zorba/projects/channeltalk-solve
LIVE      : False
임계값     : 8 (ROOTCAUSE_MIN_CONVERSATIONS)


## 1. 온톨로지 — VOC 지식그래프 스키마

원본 VOC 대화가 그래프로 정규화되는 체인:

```
Customer ──OPENS──> Conversation ──EXPRESSES──> Intent
                          │
                          └──MENTIONS──> Symptom ──IMPLICATES──> Component ──CAUSED_BY──> RootCause ──ADDRESSED_BY──> Action
```

`src/graph/schema.cypher`에 실제 제약/라벨이 정의돼 있습니다. 라벨을 추출해 봅니다.

In [2]:
import re

schema_path = config.REPO_ROOT / "src" / "graph" / "schema.cypher"
text = schema_path.read_text(encoding="utf-8")
labels = sorted(set(re.findall(r":([A-Z][A-Za-z]+)\b", text)))
print("스키마에 등장하는 노드/관계 토큰:")
for lab in labels:
    print("  •", lab)

스키마에 등장하는 노드/관계 토큰:
  • Action
  • AgentCheckpoint
  • AgentCheckpointWrite
  • Component
  • Conversation
  • Customer
  • Intent
  • RootCause
  • Symptom
  • Theme


## 2. 3중 하이브리드 검색

`retriever.hybrid_search(query, k)`는 세 개의 arm을 병렬로 돌립니다.

| arm | 방식 | 무엇을 잘 잡나 |
| :-- | :-- | :-- |
| **Dense (D)** | bge-small ONNX 임베딩 벡터 유사도 | 의미가 비슷한 대화 |
| **Sparse (S)** | Neo4j full-text (BM25) | 정확한 용어·키워드 |
| **Graph (G)** | 상위 hit이 가리키는 Component의 구조적 이웃 | 같은 근본 원인을 공유하는 대화 |

이 셀은 **라이브**(Neo4j + 임베딩)라 `LIVE=True`일 때만 실행됩니다.

In [3]:
if LIVE:
    from src.graph.retriever import hybrid_search
    res = hybrid_search("결제 실패가 왜 이렇게 많아?", k=6)
    print("top_component:", res.get("top_component"))
    for r in res["results"][:6]:
        print(f"  {r['id']:>12}  arms={r['arms']}  rrf={r['rrf']}")
else:
    print("LIVE=False → 스킵 (Neo4j Aura + fastembed 필요). LIVE=True로 실행하세요.")

LIVE=False → 스킵 (Neo4j Aura + fastembed 필요). LIVE=True로 실행하세요.


## 3. RRF — Reciprocal Rank Fusion

세 arm의 순위를 하나로 합치는 규칙 (arm 점수 스케일이 서로 달라도 **순위**만 쓰므로 안전):

$$\text{score}(d) = \sum_{\text{arm}} \frac{1}{k + \text{rank}_{\text{arm}}(d)} \qquad (k=60)$$

`retriever._rrf`를 **toy 순위 리스트**로 직접 돌려봅니다 (인프라 불필요).

In [4]:
from src.graph.retriever import _rrf

ranklists = {
    "dense":  ["conv_00007", "conv_00042", "conv_00013"],
    "sparse": ["conv_00042", "conv_00007", "conv_00099"],
    "graph":  ["conv_00007", "conv_00013"],
}
fused = _rrf(ranklists, k=60)
print("RRF 융합 랭킹 (conv_00007이 세 arm 모두에 등장 → 1위):")
for cid, score in sorted(fused.items(), key=lambda kv: -kv[1]):
    print(f"  {cid:>12}  {score:.5f}")

RRF 융합 랭킹 (conv_00007이 세 arm 모두에 등장 → 1위):
    conv_00007  0.04892
    conv_00042  0.03252
    conv_00013  0.03200
    conv_00099  0.01587


## 4. Retrieval-gating — 정직 게이트 (§5-6)

`/api/chat`은 근거 없이 답하지 않도록 **3분기**로 게이팅합니다. 경계는 루트원인 승격
임계값 `ROOTCAUSE_MIN_CONVERSATIONS`(기본 8)입니다.

| 조건 | gate | 응답 |
| :-- | :-- | :-- |
| 근거 0건 | `refuse` | 정직한 거절 + 답 가능한 질문 chips |
| 근거 有·컴포넌트 미승격(<임계값) | `low_confidence` | ⚠ 헤지 답변 (참고용) |
| 컴포넌트 승격(≥임계값) | `answer` | 실제 ₩/frequency 값으로 구성 |

`chat.py`의 분기 규칙을 순수 함수로 재현합니다 (인프라 불필요).

In [5]:
THRESH = config.ROOTCAUSE_MIN_CONVERSATIONS

def gate(n_hits: int, promoted: bool) -> str:
    if n_hits == 0:
        return "refuse"
    if not promoted:
        return "low_confidence"
    return "answer"

print(f"임계값(promotion) = {THRESH}건\n")
for n_hits, promoted in [(0, False), (3, False), (12, True)]:
    print(f"  hits={n_hits:>2}  promoted={str(promoted):<5} → {gate(n_hits, promoted)}")

임계값(promotion) = 8건

  hits= 0  promoted=False → refuse
  hits= 3  promoted=False → low_confidence
  hits=12  promoted=True  → answer


## 5. LangGraph 에이전트 + Human-in-the-Loop

`agent.build_graph(checkpointer)`는 아래 노드들을 `StateGraph`로 잇습니다. `action_drafter`
다음에서 `_needs_approval`로 분기하여 **`human_approval`에서 `interrupt()`로 실행이 멈추고**,
`Command(resume=<승인 키>)`로 재개되면 `dispatcher`가 GitHub 이슈를 발행합니다.

```mermaid
flowchart LR
  A[analyst_extract] --> B[graph_writer] --> C[researcher_cluster]
  C --> D[triage_rootcause] --> E[action_drafter]
  E -->|human| F[human_approval ⏸ interrupt] --> H[dispatcher] --> I[reporter]
  E -->|auto| G[auto_approve] --> H
```

노드 함수가 실제로 존재하는지 확인합니다 (import는 인프라 불필요 — lazy 연결).

In [6]:
from src.graph import agent

pipeline = ["analyst_extract", "graph_writer", "researcher_cluster",
            "triage_rootcause", "action_drafter", "human_approval",
            "dispatcher", "reporter"]
assert all(callable(getattr(agent, fn)) for fn in pipeline), "노드 누락"
print("파이프라인 노드 검증 OK:")
print("  " + " → ".join(pipeline))
print("\ninterrupt() 는 human_approval 에서 실행을 멈추고, Command(resume=[keys]) 로 재개됩니다.")

파이프라인 노드 검증 OK:
  analyst_extract → graph_writer → researcher_cluster → triage_rootcause → action_drafter → human_approval → dispatcher → reporter

interrupt() 는 human_approval 에서 실행을 멈추고, Command(resume=[keys]) 로 재개됩니다.


`LIVE=True`이면 아래 셀이 실제 그래프를 시작해 `interrupt` payload를 받아보고, 승인 키로
재개합니다 (Neo4j checkpointer가 콜드스타트 간 thread를 보존).

In [7]:
if LIVE:
    # 실제 왕복은 REST(/api/agent/run → /api/agent/dispatch) 또는 build_graph로 수행.
    # 데모: run → interrupt payload 확인 → dispatch(approve) 재개.
    from src.graph.checkpoint_neo4j import Neo4jCheckpointSaver
    from src.graph.agent import build_graph
    import uuid
    saver = Neo4jCheckpointSaver()
    g = build_graph(saver)
    cfg = {"configurable": {"thread_id": f"nb-{uuid.uuid4().hex[:8]}"}}
    state = g.invoke({"conversations": []}, cfg)
    print("interrupt payload:", state.get("__interrupt__"))
else:
    print("LIVE=False → 스킵 (Neo4j + OpenRouter 필요).")

LIVE=False → 스킵 (Neo4j + OpenRouter 필요).


## 정리 — 라이브 앱과의 매핑

| 노트북 개념 | 라이브 앱 |
| :-- | :-- |
| 온톨로지 | 콘솔 탭1 (OntologyGraphTab) · EvidencePanel 서브그래프 |
| 3중 하이브리드 + RRF | 콘솔 탭2 (HybridSearchTab, D/S/G 3컬럼 + RRF) |
| retrieval-gating | `/api/chat` · ChatStream 신뢰도 게이트(ConfidenceGate) |
| LangGraph interrupt | `/api/agent/run`→`dispatch` · ApprovalCard / 콘솔 탭3 |
| per-hit evidence | `ChatResponse.evidence` → EvidencePanel 근거 리스트 |

`LIVE=True` + `.env`로 각 셀을 실제 그래프에 대고 다시 실행해 보세요.